<a href="https://colab.research.google.com/github/1o1o4o5o1o4-blip/-/blob/main/Scratch_%E3%83%A6%E3%83%BC%E3%82%B6%E3%83%BC%E3%82%A2%E3%82%A4%E3%82%B3%E3%83%B3%E5%8F%96%E5%BE%97%E3%83%84%E3%83%BC%E3%83%AB%E3%81%A8%E3%82%B5%E3%83%A0%E3%83%8D%E5%8F%96%E5%BE%97%E3%83%84%E3%83%BC%E3%83%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```

```

# Scratch ユーザーアイコン　取得ツールとサムネ画像取得ツール


このノートブックは、Scratch のユーザーアイコンとプロジェクト画像を簡単に取得・表示するためのWEB 版 ツール

実行すると、まず「ユーザーアイコン」と「プロジェクト画像」のどちらを検索するかを尋ねられます。選択に応じて、ユーザー名またはプロジェクト IDかURL を入力してください。

**⚠　　username_to_searchは
choiceのuserを選択した時だけお使いください　　⚠**


In [1]:
import requests
import re
from IPython.display import Image, display, HTML

# --- 実行する機能と対象をここに設定してください (フォームフィールドを使用) ---
# 実行する機能を選択してください ('project' または 'user')
choice = 'project' # @param ['user', 'project'] {fields_with_condition: { 'user': ['username_to_search'], 'project': ['project_input_to_search'] } }
# 検索するユーザー名を入力してください
username_to_search = '' # @param {type: "string"}
# 検索するプロジェクトIDまたはURLを入力してください
project_input_to_search = 'https://scratch.mit.edu/projects/1080674870' # @param {type: "string"}


# --- ここから下のコードは変更しないでください ---

copy_js_function = """
<script>
function copyToClipboard(text, feedbackElementId) {
    navigator.clipboard.writeText(text).then(function() {
        var feedbackSpan = document.getElementById(feedbackElementId);
        if (feedbackSpan) {
            feedbackSpan.innerText = ' コピーしました！';
            feedbackSpan.style.color = 'green';
            setTimeout(function() {
                feedbackSpan.innerText = '';
            }, 2000); // Clear feedback after 2 seconds
        }
    }).catch(function(err) {
        var feedbackSpan = document.getElementById(feedbackElementId);
        if (feedbackSpan) {
            feedbackSpan.innerText = ' コピーに失敗しました: ' + err;
            feedbackSpan.style.color = 'red';
        }
        console.error('クリップボードへのコピーに失敗しました: ', err);
    });
}
</script>
"""


if choice == 'user':
    if not username_to_search:
        print("エラー: ユーザー名が設定されていません。フォームフィールドの 'username_to_search' を設定してください。")
    else:
        username = username_to_search.strip()
        api_url = f"https://api.scratch.mit.edu/users/{username}"

        try:
            response = requests.get(api_url)

            if response.status_code == 200:
                data = response.json()
                # Check for 'Too many requests' even if status code is 200
                if 'response' in data and data['response'] == 'Too many requests':
                    print("\nエラー: リクエストが多すぎます。Scratch APIのレート制限に達しました。数分待ってから再試行してください。")
                else:
                    avatar_url = data.get('profile', {}).get('images', {}).get('90x90')
                    user_id = data.get('id')

                    # If avatar_url is not directly from API or is missing, construct it.
                    # Do NOT apply percentage encoding to user avatar URLs, as it makes them inaccessible.
                    if not avatar_url and user_id:
                        # Construct the URL using a known working pattern for user avatars.
                        avatar_url = f"https://cdn2.scratch.mit.edu/get_image/user/{user_id}/90x90.png"
                    elif not avatar_url and not user_id:
                        print("エラー: アバターURLもユーザーIDも取得できませんでした。")
                        # No return here, just skip the display part below if no URL.

                    if avatar_url:
                        print(f"\n--- ユーザー「{username}」のアイコン ---")
                        display(Image(url=avatar_url))
                        display(HTML(f"画像URL: <a href='{avatar_url}' target='_blank'>{avatar_url}</a>"))

                        # Inject the JS function once per display of a button for simplicity
                        display(HTML(copy_js_function))

                        # Create unique IDs for button and feedback span
                        button_id = f"copyButton_user_{user_id}"
                        feedback_id = f"copyFeedback_user_{user_id}"

                        copy_button_html = f"""
                        <div>
                            <button id=\"{button_id}\" onclick=\"copyToClipboard('{avatar_url}', '{feedback_id}')\">画像リンクをコピー</button>
                            <span id=\"{feedback_id}\" style=\"margin-left: 10px;\"></span>
                        </div>
                        """
                        display(HTML(copy_button_html))

            else:
                # Attempt to parse JSON even if status code is not 200 for more specific error messages
                try:
                    error_data = response.json()
                    if 'response' in error_data and error_data['response'] == 'Too many requests':
                        print("\nエラー: リクエストが多すぎます。Scratch APIのレート制限に達しました。数分待ってから再試行してください。")
                    else:
                        print(f"\nエラー: 「{username}」というユーザーが見つかりませんでした。ユーザー名を確認してください。(ステータスコード: {response.status_code})")
                        print("API Response (for debugging):")
                        print(error_data)
                except requests.exceptions.JSONDecodeError:
                    print(f"\nエラー: APIからの応答を解析できませんでした。(ステータスコード: {response.status_code})")
                    print("API Response (for debugging):")
                    print(response.text)

        except Exception as e:
            print(f"\nエラーが発生しました: {e}")

elif choice == 'project':
    if not project_input_to_search:
        print("エラー: プロジェクトIDまたはURLが設定されていません。フォームフィールドの 'project_input_to_search' を設定してください。")
    else:
        project_user_input = project_input_to_search.strip()

        project_id = None
        final_image_url_to_fetch = None

        match_direct_image_url = re.search(r'(https://scratch\\.mit\\.edu/get_image/project/(\d+)_.*\\.png)', project_user_input)
        if match_direct_image_url:
            final_image_url_to_fetch = match_direct_image_url.group(1)
            project_id = match_direct_image_url.group(2)
        else:
            # Modified regex to be more general for extracting project ID
            match_project_page = re.search(r'projects/(\d+)', project_user_input)
            if match_project_page:
                project_id = match_project_page.group(1)
            elif project_user_input.isdigit():
                project_id = project_user_input

        if project_id:
            final_image_url_to_fetch = f"https://scratch.mit.edu/get_i%6d%61ge/p%72oject/{project_id}_1000x1000.png"

        if final_image_url_to_fetch and project_id:
            print(f"\n--- プロジェクトID「{project_id}」の画像URL ---")
            display(Image(url=final_image_url_to_fetch)) # Display the image
            display(HTML(f"生成された画像URL: <a href='{final_image_url_to_fetch}' target='_blank'>{final_image_url_to_fetch}</a>"))

            # Inject the JS function once per display of a button for simplicity
            display(HTML(copy_js_function))

            # Create unique IDs for button and feedback span
            button_id = f"copyButton_project_{project_id}"
            feedback_id = f"copyFeedback_project_{project_id}"

            copy_button_html = f"""
            <div>
                <button id=\"{button_id}\" onclick=\"copyToClipboard('{final_image_url_to_fetch}', '{feedback_id}')\">画像リンクをコピー</button>
                <span id=\"{feedback_id}\" style=\"margin-left: 10px;\"></span>
            </div>
            """
            display(HTML(copy_button_html))
            print("注意: このURLは直接検証されていません。ブラウザで開いて確認してください。")
        else:
            print("エラー: 有効なScratchプロジェクトIDまたはURLが見つかりませんでした。")
elif not choice:
    print("エラー: 実行する機能が設定されていません。フォームフィールドの 'choice' を 'user' または 'project' に設定してください。")
else:
    print("エラー: 無効な機能が設定されました。フォームフィールドの 'choice' を 'user' または 'project' に変更してください。")


--- プロジェクトID「1080674870」の画像URL ---


注意: このURLは直接検証されていません。ブラウザで開いて確認してください。
